
# Pipeline 2/2 — Entraînement du modèle

Second notebook du pipeline : charger le jeu nettoyé, le répartir en **80 % d'entraînement / 20 % de test**, entraîner le modèle et l'évaluer sur les données jamais vues.

Le modèle est **classique** — un gradient boosting d'arbres de décision (`HistGradientBoostingClassifier`, scikit-learn). Ni LLM, ni réseau de neurones : sur 786 lignes tabulaires, les arbres sont l'état de l'art, et leur explicabilité est exigée par le contexte santé.


In [1]:

import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src import config as cfg

warnings.filterwarnings("ignore", category=RuntimeWarning)


## 1. Chargement du jeu nettoyé

In [2]:

donnees = pd.read_parquet("outputs/donnees_propres.parquet")
colonnes_cat = Path("outputs/colonnes_categorielles.txt") \
    .read_text(encoding="utf-8").splitlines()

y = donnees.pop(cfg.TARGET)
groupes = donnees.pop("PatientID")
X = donnees
for c in colonnes_cat:
    X[c] = X[c].replace("nan", np.nan).astype("category")

print(f"{X.shape[0]} séjours × {X.shape[1]} variables — "
      f"{int(y.sum())} réadmissions ({y.mean():.1%})")


786 séjours × 132 variables — 141 réadmissions (17.9%)



## 2. Répartition 80 % entraînement / 20 % test

Une précaution indispensable : la répartition est **groupée par patient**. 238 patients ont plusieurs séjours — un découpage aléatoire simple placerait un séjour d'un patient en entraînement et un autre en test, et le modèle reconnaîtrait le patient au lieu de prédire le risque. Cette fuite gonfle les scores sans jamais se voir.

`GroupShuffleSplit` garantit qu'un patient est entièrement d'un seul côté.


In [3]:

from sklearn.model_selection import GroupShuffleSplit

decoupe = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=cfg.SEED)
i_train, i_test = next(decoupe.split(X, y, groups=groupes))

X_train, X_test = X.iloc[i_train], X.iloc[i_test]
y_train, y_test = y.iloc[i_train], y.iloc[i_test]

assert not set(groupes.iloc[i_train]) & set(groupes.iloc[i_test]), \
    "un patient est des deux côtés — fuite"

print(f"entraînement : {len(X_train)} séjours "
      f"({groupes.iloc[i_train].nunique()} patients, "
      f"{int(y_train.sum())} réadmissions, {y_train.mean():.1%})")
print(f"test         : {len(X_test)} séjours "
      f"({groupes.iloc[i_test].nunique()} patients, "
      f"{int(y_test.sum())} réadmissions, {y_test.mean():.1%})")
print("aucun patient des deux côtés : vérifié")


entraînement : 634 séjours (460 patients, 112 réadmissions, 17.7%)
test         : 152 séjours (115 patients, 29 réadmissions, 19.1%)
aucun patient des deux côtés : vérifié


## 3. Entraînement

In [4]:

from sklearn.ensemble import HistGradientBoostingClassifier

modele = HistGradientBoostingClassifier(
    categorical_features="from_dtype",
    random_state=cfg.SEED,
    max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
    l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.15,
)
modele.fit(X_train, y_train)
print(f"modèle entraîné — {modele.n_iter_} itérations "
      f"(arrêt précoce sur validation interne)")


modèle entraîné — 14 itérations (arrêt précoce sur validation interne)



## 4. Évaluation sur les 20 % jamais vus

L'exactitude est écartée : avec 18 % de réadmissions, un modèle qui répond toujours « non » atteindrait 82 %. Les métriques retenues : l'aire précision-rappel (référence = le taux de base), l'aire ROC, et la matrice de confusion au seuil de décision retenu (0,11 — un réglage de capacité du service, justifié dans le dossier technique).


In [5]:

from sklearn.metrics import (average_precision_score, confusion_matrix,
                             roc_auc_score)

proba = modele.predict_proba(X_test)[:, 1]

pr = average_precision_score(y_test, proba)
roc = roc_auc_score(y_test, proba)
print(f"PR-AUC  : {pr:.3f}   (taux de base : {y_test.mean():.3f} — "
      f"le modèle fait ×{pr / y_test.mean():.1f})")
print(f"ROC-AUC : {roc:.3f}")

pred = (proba >= cfg.DECISION_THRESHOLD).astype(int)
vn, fp, fn, vp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
print(f"\nAu seuil {cfg.DECISION_THRESHOLD} :")
print(f"  réadmissions rattrapées : {vp} / {vp + fn}  "
      f"(rappel {vp / (vp + fn):.0%})")
print(f"  alertes inutiles        : {fp}  "
      f"(précision {vp / (vp + fp):.0%})")
print(f"  sorties signalées       : {vp + fp} / {len(y_test)}")


PR-AUC  : 0.372   (taux de base : 0.191 — le modèle fait ×2.0)
ROC-AUC : 0.717

Au seuil 0.11 :
  réadmissions rattrapées : 26 / 29  (rappel 90%)
  alertes inutiles        : 94  (précision 22%)
  sorties signalées       : 120 / 152



## 5. Mise en garde sur la lecture du chiffre

Un découpage 80/20 unique ne laisse qu'environ **28 réadmissions dans le jeu de test** : le score obtenu porte une incertitude d'échantillonnage importante. C'est pourquoi le dossier technique évalue le même modèle en **validation croisée à 5 plis, groupée et stratifiée**, sur plusieurs partitions : la performance y est de **0,38 à 0,43 de PR-AUC selon la partition — environ le double du taux de base dans tous les cas**.

Le chiffre du présent notebook doit se lire à l'intérieur de cette fourchette, pas comme une mesure isolée.

**Le seuil ne se transporte pas d'un modèle à l'autre.** Au seuil 0,11, ce modèle signale **120 sorties sur 152, soit 79 %**, là où les prédictions hors échantillon du dossier technique en signalent 44 % et le modèle servi par l'application 34 %. La raison est mécanique : l'arrêt anticipé s'est déclenché ici à 14 itérations, sur un jeu de validation interne d'une centaine de séjours et d'une quinzaine de réadmissions — et un modèle arrêté si tôt produit des probabilités resserrées autour du taux de base, donc plus nombreuses à franchir 0,11. Le rappel de 90 % s'achète au prix d'une précision de 22 %.

Ce n'est pas un défaut du seuil, c'est sa nature : 0,11 est un réglage de capacité, pas une propriété du modèle. Il se recale sur la distribution des probabilités du modèle réellement déployé, à partir de la table des conséquences de la section 8 du dossier technique. Un jury qui compare les deux notebooks doit lire ce 79 % comme le symptôme d'un modèle sous-entraîné par un découpage unique, pas comme une performance.

---

**Sorties du pipeline** : le modèle entraîné et servi par l'application (`api/entrainer.py` refait cet entraînement sur la population complète et fige la traçabilité — version, seuil, importance des variables). Le dossier technique complet — audit, choix du seuil, équité, limites, journal de bord — est dans `soutenance_cisia.ipynb`.
